# DSPy Muay Thai Individual Strike Classifier

This notebook implements a DSPy-based system using BootstrapFewShotWithRandomSearch optimizer to help Google Gemini classify individual Muay Thai strikes in videos. Each video contains a single strike, and the strike type is stored in a label.txt file in each directory.


## Section 1: Setup and Configuration


In [34]:
import dspy
import google.generativeai as genai
import json
from pathlib import Path
import os
from dotenv import load_dotenv
import random
from typing import List, Dict, Tuple
import re

# Load environment variables
load_dotenv()

# Get Gemini API key
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
if not GEMINI_API_KEY:
    raise ValueError("GEMINI_API_KEY not found in environment variables. Please set it in .env file.")

# Configure Gemini
genai.configure(api_key=GEMINI_API_KEY)

# Configure DSPy with Gemini 2.5 Flash
try:
    lm = dspy.LM(model="gemini/gemini-2.5-flash", api_key=GEMINI_API_KEY)
    print('using gemini-2.5-flash')
except:
    print("Error: Failed to configure Gemini 2.5 Flash. Please check your API key and model availability.")

dspy.configure(lm=lm)

# Data paths
DATA_ROOT = Path("data/individual_strikes")

# Training split size (5 videos per directory)
TRAIN_SIZE = 5

print("Setup complete!")
print(f"Data directory: {DATA_ROOT}")
print(f"Training videos per strike type: {TRAIN_SIZE}")


using gemini-2.5-flash
Setup complete!
Data directory: data\individual_strikes
Training videos per strike type: 5


## Section 2: Data Loading


In [35]:
def load_individual_strike_examples(data_root: Path) -> List[dspy.Example]:
    """
    Load all individual strike videos from subdirectories.
    Each directory contains a label.txt file with the strike type.
    Returns a list of DSPy examples with video_path and strike_type.
    """
    examples = []
    
    # Iterate through all subdirectories in data/individual_strikes
    for strike_dir in sorted(data_root.iterdir()):
        if not strike_dir.is_dir():
            continue
        
        # Read strike type from label.txt file
        label_file = strike_dir / "label.txt"
        if not label_file.exists():
            print(f"Warning: label.txt not found in {strike_dir}, skipping...")
            continue
        
        with open(label_file, 'r') as f:
            strike_type = f.read().strip()
        
        # Get all .mov files in this directory
        video_files = sorted(strike_dir.glob("*.mov"))
        
        for video_path in video_files:
            example = dspy.Example(
                video_path=str(video_path),
                strike_type=strike_type  # Label comes from label.txt file
            ).with_inputs("video_path")
            examples.append(example)
    
    return examples

# Load all examples
all_examples = load_individual_strike_examples(DATA_ROOT)

print(f"Loaded {len(all_examples)} total examples")

# Group examples by strike type
examples_by_strike = {}
for example in all_examples:
    strike_type = example.strike_type
    if strike_type not in examples_by_strike:
        examples_by_strike[strike_type] = []
    examples_by_strike[strike_type].append(example)

print(f"\nExamples per strike type:")
for strike_type in sorted(examples_by_strike.keys()):
    count = len(examples_by_strike[strike_type])
    print(f"  {strike_type}: {count} videos")


Loaded 336 total examples

Examples per strike type:
  cross: 24 videos
  jab: 24 videos
  knee: 24 videos
  lead_elbow: 24 videos
  lead_hook: 24 videos
  lead_uppercut: 24 videos
  low_kick: 24 videos
  rear_elbow: 24 videos
  rear_hook: 24 videos
  rear_uppercut: 24 videos
  roundhouse: 24 videos
  switch_kick: 24 videos
  switch_knee: 24 videos
  teep: 24 videos


In [36]:
# Split data into train and validation
# Use the same random seed (42) as baseline notebook for reproducibility
# Select 5 videos per strike type for training, rest for validation

random.seed(42)  # Same seed as baseline notebook

train_examples = []
val_examples = []

# Split each strike type separately
for strike_type in sorted(examples_by_strike.keys()):
    strike_examples = examples_by_strike[strike_type]
    
    # Shuffle the list of videos for this strike type
    indices = list(range(len(strike_examples)))
    random.shuffle(indices)
    
    # Select first TRAIN_SIZE for training
    train_indices = set(indices[:TRAIN_SIZE])
    train_set = [strike_examples[i] for i in train_indices]
    val_set = [strike_examples[i] for i in range(len(strike_examples)) if i not in train_indices]
    
    train_examples.extend(train_set)
    val_examples.extend(val_set)
    
    print(f"{strike_type}: {len(train_set)} train, {len(val_set)} val")

print(f"\nTraining set: {len(train_examples)} examples")
print(f"Validation set: {len(val_examples)} examples")

# Display statistics
unique_strikes = set(ex.strike_type for ex in train_examples + val_examples)
print(f"\nUnique strike types: {sorted(unique_strikes)}")
print(f"Total unique strike types: {len(unique_strikes)}")


cross: 5 train, 19 val
jab: 5 train, 19 val
knee: 5 train, 19 val
lead_elbow: 5 train, 19 val
lead_hook: 5 train, 19 val
lead_uppercut: 5 train, 19 val
low_kick: 5 train, 19 val
rear_elbow: 5 train, 19 val
rear_hook: 5 train, 19 val
rear_uppercut: 5 train, 19 val
roundhouse: 5 train, 19 val
switch_kick: 5 train, 19 val
switch_knee: 5 train, 19 val
teep: 5 train, 19 val

Training set: 70 examples
Validation set: 266 examples

Unique strike types: ['cross', 'jab', 'knee', 'lead_elbow', 'lead_hook', 'lead_uppercut', 'low_kick', 'rear_elbow', 'rear_hook', 'rear_uppercut', 'roundhouse', 'switch_kick', 'switch_knee', 'teep']
Total unique strike types: 14


## Section 3: DSPy Signature and Module


In [37]:
class StrikeClassifier(dspy.Signature):
    """Classify the single Muay Thai strike in a video. 
    
    Watch the video carefully and analyze the movement patterns:
    - For punches: Observe hand position, arm trajectory, and body rotation
    - For kicks: Observe leg movement, angle of attack, and foot position  
    - For elbows: Observe arm position and striking angle
    - For knees: Observe leg lift and driving motion
    - For teep: Observe forward push motion (different from roundhouse)
    
    Pay attention to:
    - Which limb is used (lead vs rear)
    - The direction and trajectory of the strike
    - Body positioning and rotation
    
    The strike must be one of: cross, jab, knee, lead_elbow, lead_hook, lead_uppercut, low_kick, rear_elbow, rear_hook, rear_uppercut, roundhouse, switch_kick, switch_knee, teep.
    
    Return just the strike name (e.g., 'jab', 'cross', 'lead_hook')."""
    video_path: str = dspy.InputField(desc="Path to video file containing a single Muay Thai strike")
    strike: str = dspy.OutputField(desc="Single strike name matching one of the valid strike types")


In [38]:
class MuayThaiStrikeClassifier(dspy.Module):
    def __init__(self):
        super().__init__()
        self.classifier = dspy.ChainOfThought(StrikeClassifier)
    
    def forward(self, video_path: str):
        return self.classifier(video_path=video_path)

# Initialize classifier
classifier = MuayThaiStrikeClassifier()

print("Classifier module created!")


Classifier module created!


In [39]:
# Test basic classification on a sample video before optimization
# Note: This requires Gemini to process video, which may take some time

test_example = train_examples[0]
print(f"Testing on: {test_example.video_path}")
print(f"True label: {test_example.strike_type}")

try:
    result = classifier(video_path=test_example.video_path)
    print(f"Predicted: {result.strike}")
except Exception as e:
    print(f"Error during classification: {e}")
    print("This is expected if Gemini video API requires different setup. Continuing with optimizer...")


Testing on: data\individual_strikes\a\14.mov
True label: cross
Predicted: roundhouse


## Section 4: BootstrapFewShotWithRandomSearch Optimizer


In [40]:
def normalize_strike_name(strike: str) -> str:
    """
    Normalize strike name to match directory naming convention.
    Handles variations like spaces, hyphens, case differences.
    """
    # Convert to lowercase and replace spaces/hyphens with underscores
    normalized = strike.lower().strip()
    normalized = normalized.replace(' ', '_').replace('-', '_')
    return normalized

def exact_match_metric(gold, pred, trace=None):
    """Evaluate exact match between predicted and true strike."""
    # Get predicted strike
    try:
        predicted_strike = pred.strike if hasattr(pred, 'strike') else str(pred)
    except (AttributeError, TypeError):
        predicted_strike = ""
    
    # Get true strike
    true_strike = gold.strike_type if hasattr(gold, 'strike_type') else str(gold)
    
    # Normalize both for comparison
    true_normalized = normalize_strike_name(true_strike)
    pred_normalized = normalize_strike_name(predicted_strike)
    
    # Check exact match
    return true_normalized == pred_normalized

def soft_accuracy_metric(gold, pred, trace=None):
    """Give partial credit for similar strikes."""
    try:
        predicted_strike = pred.strike if hasattr(pred, 'strike') else str(pred)
    except (AttributeError, TypeError):
        predicted_strike = ""
    
    true_strike = gold.strike_type if hasattr(gold, 'strike_type') else str(gold)
    
    true_normalized = normalize_strike_name(true_strike)
    pred_normalized = normalize_strike_name(predicted_strike)
    
    # Exact match
    if true_normalized == pred_normalized:
        return 1.0
    
    # Partial credit for similar strikes
    similar_groups = [
        {'jab', 'cross'},  # Both straight punches
        {'lead_hook', 'rear_hook'},  # Both hooks
        {'lead_uppercut', 'rear_uppercut'},  # Both uppercuts
        {'roundhouse', 'switch_kick'},  # Both roundhouse kicks
        {'knee', 'switch_knee'},  # Both knees
        {'lead_elbow', 'rear_elbow'},  # Both elbows
    ]
    
    for group in similar_groups:
        if true_normalized in group and pred_normalized in group:
            return 0.5
    
    return 0.0

print("Evaluation metrics defined!")
print("Available metrics: exact_match_metric, soft_accuracy_metric")


Evaluation metrics defined!
Available metrics: exact_match_metric, soft_accuracy_metric


In [41]:
# Configure BootstrapFewShotWithRandomSearch optimizer
# This does a thorough search across training examples
# Using this instead of MIPROv2 to avoid parameter conflicts
from dspy.teleprompt import BootstrapFewShotWithRandomSearch

optimizer = BootstrapFewShotWithRandomSearch(
    metric=exact_match_metric,
    max_bootstrapped_demos=16,  # Increased from 8 for better optimization
    max_labeled_demos=32,       # Increased from 16 for more examples
    num_candidate_programs=20,   # Increased from 10 for better search
    num_threads=1  # Sequential processing for video API calls
)

print("Optimizer configured!")
print(f"Optimizer: BootstrapFewShotWithRandomSearch")
print(f"Max bootstrapped demos: 16")
print(f"Max labeled demos: 32")
print(f"Number of candidate programs: 20")
print(f"Metric: exact_match_metric")
print("This will take longer but will properly optimize by evaluating more examples.")


Going to sample between 1 and 16 traces per predictor.
Will attempt to bootstrap 20 candidate sets.
Optimizer configured!
Optimizer: BootstrapFewShotWithRandomSearch
Max bootstrapped demos: 16
Max labeled demos: 32
Number of candidate programs: 20
Metric: exact_match_metric
This will take longer but will properly optimize by evaluating more examples.


In [42]:
# Run optimization on training set
# This may take a while as it needs to process videos through Gemini API

print("Starting optimization...")
print(f"Training on {len(train_examples)} examples")

try:
    optimized_classifier = optimizer.compile(
        student=classifier,
        trainset=train_examples
    )
    print("\nOptimization complete!")
except Exception as e:
    print(f"Error during optimization: {e}")
    print("This may be due to API rate limits or video processing issues.")
    print("You may need to adjust the optimizer parameters or reduce the training set size.")
    raise


Starting optimization...
Training on 70 examples
Average Metric: 3.00 / 70 (4.3%): 100%|██████████| 70/70 [02:36<00:00,  2.24s/it] 

2025/11/23 19:25:24 INFO dspy.evaluate.evaluate: Average Metric: 3 / 70 (4.3%)



New best score: 4.29 for seed -3
Scores so far: [4.29]
Best score so far: 4.29
Average Metric: 3.00 / 70 (4.3%): 100%|██████████| 70/70 [00:00<00:00, 120.26it/s]

2025/11/23 19:25:25 INFO dspy.evaluate.evaluate: Average Metric: 3 / 70 (4.3%)



Scores so far: [4.29, 4.29]
Best score so far: 4.29


...


Scores so far: [4.29, 4.29, 12.86, 11.43, 14.29, 14.29, 14.29, 17.14, 15.71, 14.29, 15.71, 14.29, 17.14, 10.0, 15.71, 15.71, 12.86, 15.71, 14.29, 14.29, 14.29, 15.71]
Best score so far: 17.14


 69%|██████▊   | 48/70 [00:01<00:00, 32.22it/s]


Bootstrapped 2 full traces after 48 examples for up to 1 rounds, amounting to 48 attempts.
Average Metric: 10.00 / 70 (14.3%): 100%|██████████| 70/70 [02:19<00:00,  1.99s/it]

2025/11/23 19:50:31 INFO dspy.evaluate.evaluate: Average Metric: 10 / 70 (14.3%)



Scores so far: [4.29, 4.29, 12.86, 11.43, 14.29, 14.29, 14.29, 17.14, 15.71, 14.29, 15.71, 14.29, 17.14, 10.0, 15.71, 15.71, 12.86, 15.71, 14.29, 14.29, 14.29, 15.71, 14.29]
Best score so far: 17.14
23 candidate programs found.

Optimization complete!


In [43]:
# Display selected examples
if hasattr(optimized_classifier.classifier, '_compiled_demos'):
    print("Selected few-shot examples:")
    for i, demo in enumerate(optimized_classifier.classifier._compiled_demos[:8], 1):
        print(f"\nExample {i}:")
        print(f"  Video: {Path(demo.video_path).name}")
        print(f"  Strike: {demo.strike_type}")


## Section 5: Evaluation


In [44]:
# Evaluate optimized model on validation set
print(f"Evaluating on {len(val_examples)} validation examples...")

results = []
exact_matches = 0

for i, example in enumerate(val_examples):
    try:
        prediction = optimized_classifier(video_path=example.video_path)
        
        # Extract predicted strike
        try:
            predicted_strike = prediction.strike if hasattr(prediction, 'strike') else str(prediction)
        except (AttributeError, TypeError):
            predicted_strike = ""
        
        # Normalize for comparison
        true_strike = example.strike_type
        pred_normalized = normalize_strike_name(predicted_strike)
        true_normalized = normalize_strike_name(true_strike)
        
        exact_match = (true_normalized == pred_normalized)
        
        if exact_match:
            exact_matches += 1
        
        results.append({
            'video_path': example.video_path,
            'true_strike': true_strike,
            'predicted_strike': predicted_strike,
            'exact_match': exact_match
        })
        
        if (i + 1) % 20 == 0:
            print(f"Processed {i + 1}/{len(val_examples)} examples...")
            
    except Exception as e:
        print(f"Error processing {example.video_path}: {e}")
        results.append({
            'video_path': example.video_path,
            'true_strike': example.strike_type,
            'predicted_strike': "",
            'exact_match': False,
            'error': str(e)
        })

print("\nEvaluation complete!")


Evaluating on 266 validation examples...
Processed 20/266 examples...
Processed 40/266 examples...
Processed 60/266 examples...
Processed 80/266 examples...
Processed 100/266 examples...
Processed 120/266 examples...
Processed 140/266 examples...
Processed 160/266 examples...
Processed 180/266 examples...
Processed 200/266 examples...
Processed 220/266 examples...
Processed 240/266 examples...
Processed 260/266 examples...

Evaluation complete!


In [45]:
# Calculate and display metrics
exact_match_accuracy = exact_matches / len(val_examples) if val_examples else 0

print("=" * 60)
print("EVALUATION RESULTS")
print("=" * 60)
print(f"Exact Match Accuracy: {exact_match_accuracy:.2%} ({exact_matches}/{len(val_examples)})")
print()

# Display some example predictions
print("Sample Predictions:")
print("-" * 60)
for i, result in enumerate(results[:10], 1):
    print(f"\nExample {i}:")
    print(f"  Video: {Path(result['video_path']).name}")
    print(f"  True:  {result['true_strike']}")
    print(f"  Pred:  {result['predicted_strike']}")
    print(f"  Match: {'✓' if result['exact_match'] else '✗'}")


EVALUATION RESULTS
Exact Match Accuracy: 14.29% (38/266)

Sample Predictions:
------------------------------------------------------------

Example 1:
  Video: 1.mov
  True:  cross
  Pred:  cross
  Match: ✓

Example 2:
  Video: 10.mov
  True:  cross
  Pred:  jab
  Match: ✗

Example 3:
  Video: 11.mov
  True:  cross
  Pred:  cross
  Match: ✓

Example 4:
  Video: 12.mov
  True:  cross
  Pred:  jab
  Match: ✗

Example 5:
  Video: 13.mov
  True:  cross
  Pred:  jab
  Match: ✗

Example 6:
  Video: 15.mov
  True:  cross
  Pred:  jab
  Match: ✗

Example 7:
  Video: 16.mov
  True:  cross
  Pred:  cross
  Match: ✓

Example 8:
  Video: 17.mov
  True:  cross
  Pred:  jab
  Match: ✗

Example 9:
  Video: 18.mov
  True:  cross
  Pred:  jab
  Match: ✗

Example 10:
  Video: 19.mov
  True:  cross
  Pred:  cross
  Match: ✓


In [46]:
# Calculate per-strike-type accuracy
from collections import defaultdict

strike_stats = defaultdict(lambda: {'correct': 0, 'total': 0, 'predicted': 0})

for result in results:
    true_strike = result['true_strike']
    pred_strike = result['predicted_strike']
    
    # Count true strikes
    strike_stats[true_strike]['total'] += 1
    
    # Count predicted strikes (if not empty)
    if pred_strike:
        pred_normalized = normalize_strike_name(pred_strike)
        strike_stats[pred_normalized]['predicted'] += 1
    
    # Count correct predictions
    if result['exact_match']:
        strike_stats[true_strike]['correct'] += 1

print("\nPer-Strike-Type Accuracy:")
print("-" * 60)
for strike in sorted(strike_stats.keys()):
    stats = strike_stats[strike]
    accuracy = stats['correct'] / stats['total'] if stats['total'] > 0 else 0
    print(f"  {strike:20s}: {accuracy:6.2%} ({stats['correct']}/{stats['total']} correct, {stats['predicted']} predicted)")



Per-Strike-Type Accuracy:
------------------------------------------------------------
  cross               : 31.58% (6/19 correct, 10 predicted)
  jab                 : 47.37% (9/19 correct, 51 predicted)
  knee                : 21.05% (4/19 correct, 57 predicted)
  lead_elbow          :  0.00% (0/19 correct, 14 predicted)
  lead_hook           : 10.53% (2/19 correct, 49 predicted)
  lead_uppercut       :  0.00% (0/19 correct, 6 predicted)
  low_kick            :  5.26% (1/19 correct, 13 predicted)
  rear_elbow          :  5.26% (1/19 correct, 3 predicted)
  rear_hook           :  5.26% (1/19 correct, 7 predicted)
  rear_uppercut       :  0.00% (0/19 correct, 3 predicted)
  roundhouse          : 26.32% (5/19 correct, 42 predicted)
  switch_kick         :  0.00% (0/19 correct, 1 predicted)
  switch_knee         :  0.00% (0/19 correct, 0 predicted)
  teep                : 47.37% (9/19 correct, 10 predicted)


## Section 6: Results and Analysis


In [47]:
# Save evaluation results
output_dir = Path("results")
output_dir.mkdir(exist_ok=True)

# Save evaluation results
results_data = {
    'metrics': {
        'exact_match_accuracy': exact_match_accuracy,
        'total_examples': len(val_examples),
        'exact_matches': exact_matches
    },
    'results': results,
    'strike_stats': {k: dict(v) for k, v in strike_stats.items()}
}

with open(output_dir / "individual_strikes_dspy.json", 'w') as f:
    json.dump(results_data, f, indent=2)

print(f"Results saved to {output_dir / 'individual_strikes_dspy.json'}")


Results saved to results\individual_strikes_dspy.json


In [48]:
# Display summary
print("\n" + "=" * 60)
print("SUMMARY")
print("=" * 60)
print(f"Model: Gemini 2.5 Flash")
print(f"Optimizer: BootstrapFewShotWithRandomSearch")
print(f"Dataset: Individual Strikes (one strike per video)")
print(f"Training examples: {len(train_examples)} ({TRAIN_SIZE} per strike type)")
print(f"Validation examples: {len(val_examples)}")
print(f"\nFinal Performance:")
print(f"  Exact Match Accuracy: {exact_match_accuracy:.2%}")
print(f"\nResults saved to: {output_dir}/individual_strikes_dspy.json")
print("=" * 60)



SUMMARY
Model: Gemini 2.5 Flash
Optimizer: BootstrapFewShotWithRandomSearch
Dataset: Individual Strikes (one strike per video)
Training examples: 70 (5 per strike type)
Validation examples: 266

Final Performance:
  Exact Match Accuracy: 14.29%

Results saved to: results/individual_strikes_dspy.json
